### Data Cleaning Olist - Chuẩn bị dữ liệu cho phân tích
##### Yêu cầu: Dữ liệu đầu ra phải chính xác tuyệt đối, không có lỗi logic

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from churn_prediction.paths import SRC_DIR, PROCESSED_DIR, INTERIM_NOTEBOOK_DIR, RAW_DIR

In [4]:
orders = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv')
customers = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
order_items = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW_DIR / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(RAW_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW_DIR / 'olist_sellers_dataset.csv')

In [5]:
print("\nTHIẾT LẬP VALIDATION RULES")

VALIDATION_RULES = {
    # Orders validation
    'order_status_valid': ['delivered', 'shipped', 'processing', 'approved', 'invoiced'],
    'order_status_invalid': ['canceled', 'unavailable'],
    
    # Payment validation
    'payment_type_valid': ['credit_card', 'boleto', 'voucher', 'debit_card'],
    'min_payment_value': 0.01,
    'max_payment_installments': 24,
    
    # Review validation
    'review_score_valid': [1, 2, 3, 4, 5],
    
    # Price validation
    'min_price': 0.01,
    'min_freight': 0.0,
    
    # Temporal validation
    'max_days_to_delivery': 365,
    'max_days_to_review': 180
}

print("   Đã thiết lập validation rules")


THIẾT LẬP VALIDATION RULES
   Đã thiết lập validation rules


In [6]:
# Danh sách file đã đọc
datasets = {
    'orders': orders.shape,
    'customers': customers.shape,
    'order_items': order_items.shape,
    'payments': payments.shape,
    'reviews': reviews.shape,
    'products': products.shape,
    'sellers': sellers.shape
}

print("\nKích thước các bảng:")
for name, shape in datasets.items():
    print(f"  {name}: {shape[0]:,} dòng x {shape[1]} cột")


Kích thước các bảng:
  orders: 99,441 dòng x 8 cột
  customers: 99,441 dòng x 5 cột
  order_items: 112,650 dòng x 7 cột
  payments: 103,886 dòng x 5 cột
  reviews: 99,224 dòng x 7 cột
  products: 32,951 dòng x 9 cột
  sellers: 3,095 dòng x 4 cột


### xử lý bảng orders

In [ ]:
print("1. XỬ LÝ BẢNG ORDERS")
# Xem thông tin
print("\nThông tin orders:")
orders.info()

1. XỬ LÝ BẢNG ORDERS

Thông tin orders:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [7]:
# 1.1. Chuyển đổi cột thời gian
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

In [8]:
# 1.2. VALIDATION: Kiểm tra logic thời gian
print("\n   Kiểm tra logic thời gian:")
invalid_dates = orders[orders['order_approved_at'] < orders['order_purchase_timestamp']]
print(f"   - Đơn có approved_at < purchase_at: {len(invalid_dates)}")
orders.loc[invalid_dates.index, 'order_approved_at'] = orders.loc[invalid_dates.index, 'order_purchase_timestamp']



   Kiểm tra logic thời gian:
   - Đơn có approved_at < purchase_at: 0


In [9]:
# 1.3. Xử lý missing values
orders['order_approved_at'] = orders['order_approved_at'].fillna(orders['order_purchase_timestamp'])


In [10]:
# 1.4. Lọc đơn hàng hợp lệ (QUAN TRỌNG: loại bỏ đơn bị hủy)
valid_status = ['delivered', 'shipped', 'processing', 'approved', 'invoiced']
orders_clean = orders[orders['order_status'].isin(valid_status)].copy()
print(f"   - Đơn hàng hợp lệ: {len(orders_clean):,} / {len(orders):,} (bỏ {len(orders)-len(orders_clean):,} đơn)")

   - Đơn hàng hợp lệ: 98,202 / 99,441 (bỏ 1,239 đơn)


In [12]:
# 1.5. Tạo features thời gian
orders_clean['purchase_year'] = orders_clean['order_purchase_timestamp'].dt.year
orders_clean['purchase_month'] = orders_clean['order_purchase_timestamp'].dt.month
orders_clean['purchase_weekday'] = orders_clean['order_purchase_timestamp'].dt.dayofweek
orders_clean['purchase_hour'] = orders_clean['order_purchase_timestamp'].dt.hour

### xử lý bảng customers

In [53]:
print("2. XỬ LÝ BẢNG CUSTOMERS")
print("\nThông tin customers:")
customers.info()
# Kiểm tra duplicate customer_id
duplicate_customers = customers['customer_id'].duplicated().sum()
print(f"\nCustomer_id bị duplicate: {duplicate_customers}")
# Kiểm tra missing
print(f"\nMissing values: {customers.isnull().sum().sum()}")



2. XỬ LÝ BẢNG CUSTOMERS

Thông tin customers:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

Customer_id bị duplicate: 0

Missing values: 0


### XỬ LÝ BẢNG ORDER_ITEMS

In [14]:
print("3. XỬ LÝ BẢNG ORDER_ITEMS")
print("\nThông tin order_items:")
order_items.info()


3. XỬ LÝ BẢNG ORDER_ITEMS

Thông tin order_items:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [13]:
# 3.1. VALIDATION: Chỉ giữ orders đã được xác thực
order_items = order_items[order_items['order_id'].isin(orders_clean['order_id'])].copy()
print(f"   - Order_items sau khi lọc: {len(order_items):,}")

   - Order_items sau khi lọc: 112,101


In [16]:
# 3.2. Xử lý outlier price (cắt bỏ top 1% để tránh nhiễu)
q99_price = order_items['price'].quantile(0.99)
price_outliers = order_items['price'] > q99_price
print(f"   - Price outliers (>{q99_price:.2f}): {price_outliers.sum():,}")
order_items.loc[price_outliers, 'price'] = q99_price

   - Price outliers (>889.00): 1,116


In [54]:
# 3.3. Xử lý freight_value <= 0
negative_freight = order_items['freight_value'] <= 0
print(f"   - Freight <= 0: {negative_freight.sum():,}")
median_freight = order_items[order_items['freight_value'] > 0]['freight_value'].median()
order_items.loc[negative_freight, 'freight_value'] = median_freight



   - Freight <= 0: 0


### Xử lý bảng  PAYMENTS

In [17]:
print("4. XỬ LÝ BẢNG PAYMENTS")
print("\nThông tin payments:")
payments.info()

4. XỬ LÝ BẢNG PAYMENTS

Thông tin payments:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [19]:
# 4.1. VALIDATION: Chỉ giữ orders đã xác thực
payments = payments[payments['order_id'].isin(orders_clean['order_id'])].copy()
print(f"   - Payments sau khi lọc: {len(payments):,}")

   - Payments sau khi lọc: 102,568


In [20]:
# 4.2. VALIDATION: Payment_value phải > 0
invalid_payment = payments['payment_value'] <= 0
print(f"   - Payment <= 0: {invalid_payment.sum():,}")
payments.loc[invalid_payment, 'payment_value'] = payments['payment_value'].median()

   - Payment <= 0: 6


In [55]:
# 4.3. VALIDATION: Payment_installments hợp lý
invalid_installments = payments['payment_installments'] < 1
payments.loc[invalid_installments, 'payment_installments'] = 1

In [56]:
# 4.4. VALIDATION: Payment_type hợp lệ
invalid_type = ~payments['payment_type'].isin(VALIDATION_RULES['payment_type_valid'])
print(f"   - Payment_type không hợp lệ: {invalid_type.sum():,}")
payments.loc[invalid_type, 'payment_type'] = 'other'

   - Payment_type không hợp lệ: 0


In [24]:
# 4.5. Gộp payments theo order (vì 1 order có thể nhiều payment)
payments_grouped = payments.groupby('order_id').agg({
    'payment_value': 'sum',
    'payment_installments': 'max',
    'payment_type': lambda x: x.mode()[0] if len(x) > 0 else 'unknown'
}).reset_index()

print(f"   - Số order sau khi gộp payments: {len(payments_grouped):,}")

   - Số order sau khi gộp payments: 98,201


### xử lý bảng REVIEWS

In [25]:
print("5. XỬ LÝ BẢNG REVIEWS")
print("\nThông tin reviews:")
reviews.info()

5. XỬ LÝ BẢNG REVIEWS

Thông tin reviews:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [26]:
# 5.1. Chỉ giữ orders đã xác thực
reviews = reviews[reviews['order_id'].isin(orders_clean['order_id'])].copy()
print(f"   - Reviews sau khi lọc: {len(reviews):,}")

   - Reviews sau khi lọc: 98,015


In [27]:
# 5.2. Chuyển đổi thời gian
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

In [28]:
# 5.3. Tính thời gian phản hồi
reviews['days_to_answer'] = (reviews['review_answer_timestamp'] - reviews['review_creation_date']).dt.days

In [29]:
# 5.4. VALIDATION: Review_score hợp lệ
invalid_score = ~reviews['review_score'].isin(VALIDATION_RULES['review_score_valid'])
print(f"   - Review_score không hợp lệ: {invalid_score.sum():,}")
reviews.loc[invalid_score, 'review_score'] = 3

   - Review_score không hợp lệ: 0


In [31]:
# 5.5. TẠO CỘT has_comment 
reviews['has_comment'] = reviews['review_comment_message'].notna().astype(int)
print(f"   - Tỉ lệ có comment: {reviews['has_comment'].mean()*100:.1f}%")

   - Tỉ lệ có comment: 41.0%


In [32]:
# 5.5. Gộp reviews theo order (mỗi order có thể nhiều review?)
reviews_grouped = reviews.groupby('order_id').agg({
    'review_score': 'mean',
    'days_to_answer': 'mean',
    'has_comment': 'sum'
}).reset_index()
reviews_grouped.columns = ['order_id', 'review_score', 'days_to_answer', 'num_comments']

print(f"   - Số order sau khi gộp reviews: {len(reviews_grouped):,}")

   - Số order sau khi gộp reviews: 97,470


In [57]:
# 5.7. Xử lý missing (order không có review)
reviews_grouped['review_score'] = reviews_grouped['review_score'].fillna(3)  # neutral
reviews_grouped['days_to_answer'] = reviews_grouped['days_to_answer'].fillna(0)
reviews_grouped['num_comments'] = reviews_grouped['num_comments'].fillna(0).astype(int)

print(f"   - Số order sau khi gộp reviews: {len(reviews_grouped):,}")

   - Số order sau khi gộp reviews: 97,470


### xử lý bảng PRODUCTS

In [34]:
print("6. XỬ LÝ BẢNG PRODUCTS")
print("\nThông tin products:")
products.info()

6. XỬ LÝ BẢNG PRODUCTS

Thông tin products:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [35]:
# 6.1. Kiểm tra missing
print("\nMissing values trong products:")
missing_products = products.isnull().sum()
missing_products_pct = 100 * missing_products / len(products)
missing_products_df = pd.DataFrame({'Missing': missing_products, 'Percentage': missing_products_pct})
print(missing_products_df[missing_products_df['Missing'] > 0])


Missing values trong products:
                            Missing  Percentage
product_category_name           610    1.851234
product_name_lenght             610    1.851234
product_description_lenght      610    1.851234
product_photos_qty              610    1.851234
product_weight_g                  2    0.006070
product_length_cm                 2    0.006070
product_height_cm                 2    0.006070
product_width_cm                  2    0.006070


In [36]:
# 6.2. Xử lý product_category_name (fill 'unknown' cho missing)
products['product_category_name'] = products['product_category_name'].fillna('unknown')


In [58]:
# 6.3. Xử lý kích thước sản phẩm (fill 0 cho missing)
dimension_cols = ['product_name_length', 'product_description_length', 'product_photos_qty',
                  'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in dimension_cols:
    if col in products.columns:
        products[col] = products[col].fillna(0)



### xử lý dữ liệu sellers

In [50]:
print("7. XỬ LÝ BẢNG SELLERS")

print("\nThông tin sellers:")
sellers.info()

# Kiểm tra missing
print(f"\nMissing values: {sellers.isnull().sum().sum()}")



7. XỬ LÝ BẢNG SELLERS

Thông tin sellers:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB

Missing values: 0


### LƯU DỮ LIỆU ĐÃ CLEAN

In [59]:
print("LƯU DỮ LIỆU ĐÃ CLEAN")

# Lưu từng bảng
orders.to_parquet(PROCESSED_DIR / 'orders_clean.parquet', index=False)
customers.to_parquet(PROCESSED_DIR / 'customers_clean.parquet', index=False)
order_items.to_parquet(PROCESSED_DIR / 'order_items_clean.parquet', index=False)
payments.to_parquet(PROCESSED_DIR / 'payments_clean.parquet', index=False)
reviews.to_parquet(PROCESSED_DIR / 'reviews_clean.parquet', index=False)
products.to_parquet(PROCESSED_DIR / 'products_clean.parquet', index=False)
sellers.to_parquet(PROCESSED_DIR / 'sellers_clean.parquet', index=False)

print("Đã lưu 7 bảng dữ liệu đã clean vào PROCESSED_DIR")
print("\nDATA CLEANING HOÀN TẤT!")

LƯU DỮ LIỆU ĐÃ CLEAN
Đã lưu 7 bảng dữ liệu đã clean vào PROCESSED_DIR

DATA CLEANING HOÀN TẤT!
